In [ ]:
import pandas as pd
import numpy as np

from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, concat_ws
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt


# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
-- Welcome to Snowflake Notebooks!
-- Try out a SQL cell to generate some data.
SELECT * 
FROM SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_L3_SUB
WHERE PRICING_STATUS_C = 'Priced'

In [ ]:
# Then, we can use the python name to turn cell2 into a Pandas dataframe
df = cell2.to_pandas()



In [ ]:
X = np.vstack(df["PRODUCT_EMBED_V2"].values)
X.shape

In [ ]:
pca = PCA(n_components=100, random_state=42)
X_pca = pca.fit_transform(X)
X_pca.shape

np.sum(pca.explained_variance_ratio_)


In [ ]:
from sklearn.utils import resample

SAMPLE_SIZE = 30000

X_sample = resample(
    X_pca,
    n_samples=SAMPLE_SIZE,
    random_state=42
)

In [ ]:
ks = [15, 16, 17, 18, 19, 20]
sil_scores = {}

for k in ks:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto",
        max_iter=200
    )
    labels = km.fit_predict(X_sample)
    sil_scores[k] = silhouette_score(X_sample, labels)

sil_scores


In [ ]:
best_k = max(sil_scores, key=sil_scores.get)
best_k


In [ ]:
k_final = 19  # example

km_final = KMeans(
    n_clusters=k_final,
    random_state=42,
    n_init="auto",
    max_iter=300
)

full_labels = km_final.fit_predict(X_pca)
df["CLUSTER"] = full_labels.astype("int32")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS



DOMAIN_STOPWORDS = {
    "price", "pricing",
    "status",
    "quote", "quoted",
    "description",
    "insert",
    "item", "items",
    "product", "products",
    "unit", "units",
    # "available",
    # "order", "ordering",
    # "new", "old"
    }

STOPWORDS = list(ENGLISH_STOP_WORDS.union(DOMAIN_STOPWORDS))

vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words=STOPWORDS,
    ngram_range=(1, 2),
    min_df=5, 
    max_df=0.8 
)

tfidf_matrix = vectorizer.fit_transform(df["TEXT_TO_EMBED"])
feature_names = np.array(vectorizer.get_feature_names_out())



In [ ]:
def top_terms_per_cluster(tfidf, df, feature_names, top_n=15):
    cluster_terms = {}

    for c in sorted(df["CLUSTER"].unique()):
        mask = (df["CLUSTER"] == c).values  # numpy bool array

        centroid = tfidf[mask].mean(axis=0)
        centroid = np.asarray(centroid).ravel()

        top_idx = centroid.argsort()[::-1][:top_n]
        cluster_terms[int(c)] = feature_names[top_idx].tolist()

    return cluster_terms


In [ ]:
cluster_keywords = top_terms_per_cluster(
    tfidf=tfidf_matrix,
    df=df,
    feature_names=feature_names,
    top_n=15
)

In [ ]:
cluster_keywords

1. What this analysis is really telling you

Across these 19 clusters, the dominant signal is product modality, not chemistry per se:

Small-molecule chemicals (reagents, acids, substituted compounds)

Proteins / antibodies / recombinant products

Kits (ELISA, siRNA, shRNA)

Biospecimens (plasma, serum, patient samples)

Commercial metadata (availability, lead time, vendor, MSDS)

Packaging & quantity conventions (mg, g, ea, pk)

This is exactly what you want at a higher taxonomy level: clusters are separating by how the product is sold and used, not fine scientific semantics.

So yes — this supports collapsing 22 → ~8 top-level categories, with finer levels underneath.

Chemical Reagents (Small Molecules)

Proteins & Recombinant Products

Antibodies

Kits & Assays

Human Biospecimens

Chemical Identity & Specifications

Packaging & Quantity Information

Availability, Shipping & Compliance

In [ ]:
# Collapse k=19 clusters (0–18) into 7 semantic groups
# Collapse k=19 clusters (0–18) into 7 semantic groups

CLUSTER_TO_GROUP = {
    # 0 — Small-Molecule Chemicals & Compounds
    0: 0,
    4: 0,
    5: 0,
    6: 0,
    7: 0,
    14: 0,
    15: 0,
    18: 0,

    # 1 — Antibodies
    1: 1,
    16: 1,

    # 2 — Proteins & Recombinant Biomolecules
    2: 2,

    # 3 — Nucleic Acids & Genetic Constructs
    3: 3,
    17: 3,

    # 4 — Assay & Detection Kits
    10: 4,

    # 5 — Clinical & Biological Samples
    9: 5,

    # 6 — Laboratory Consumables & Packaged Items
    8: 6,
    11: 6,
    12: 6,
    13: 6,
}


df["GROUP"] = df["CLUSTER"].map(CLUSTER_TO_GROUP)

In [ ]:
unmapped_clusters = df[df["GROUP"].isna()]["CLUSTER"].unique()
if len(unmapped_clusters) > 0:
    raise ValueError(
        f"Unmapped CLUSTER values found: {sorted(unmapped_clusters)}"
    )

In [ ]:
unmapped_clusters = (
    df.loc[df["GROUP"].isna(), "CLUSTER"]
      .dropna()
      .unique()
)

if len(unmapped_clusters) > 0:
    print("❌ Unmapped CLUSTER values detected:")
    print(sorted(unmapped_clusters))

    # ----------------------------------------
    # DIAGNOSTIC STEP A: SIZE OF EACH CLUSTER
    # ----------------------------------------
    print("\nCluster sizes:")
    print(df["CLUSTER"].value_counts().loc[unmapped_clusters])

    # ----------------------------------------
    # DIAGNOSTIC STEP B: SAMPLE RAW TEXT
    # ----------------------------------------
    print("\nSample records for diagnosis:")
    sample_df = (
        df[df["CLUSTER"].isin(unmapped_clusters)]
        .sample(min(20, len(df)), random_state=42)
    )

    print(sample_df[["CLUSTER", "TEXT_TO_EMBED"]])

    # ----------------------------------------
    # STOP PIPELINE — FIX TAXONOMY DECISION
    # ----------------------------------------
    raise ValueError(
        "Unmapped clusters found. "
        "Inspect samples above and update CLUSTER_TO_GROUP mapping."
    )

In [ ]:
GROUP_TO_L3 = {
    0: "Small-Molecule Chemicals & Compounds",
    1: "Antibodies",
    2: "Proteins & Recombinant Biomolecules",
    3: "Nucleic Acids & Genetic Constructs",
    4: "Assay & Detection Kits",
    5: "Clinical & Biological Samples",
    6: "Laboratory Consumables & Packaged Items",
}

df["PROPOSED_L3"] = df["GROUP"].map(GROUP_TO_L3)

In [ ]:
assert df["PROPOSED_L3"].isna().sum() == 0

In [ ]:
# create Snowpark DataFrame
snow_df = session.create_dataframe(df)

# write to table L3_PROPOSED (overwrite if exists)
snow_df.write.mode("overwrite").save_as_table("PROPOSED_L3")